In [1]:
# Core data handling and plotting
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Best tabular model from the project
from sklearn.ensemble import GradientBoostingRegressor

# Interactive controls for the forecast demo
from ipywidgets import interact, IntSlider, Dropdown

In [2]:
# Load the feature-engineered training data for model fitting
train_features = pd.read_csv("../data/processed/train_features.csv")

# Load the full hourly demand dataset to provide historical seed values
hourly_df = pd.read_csv("../data/processed/hourly_demand.csv")

# Convert timestamps to datetime
train_features["hour"] = pd.to_datetime(train_features["hour"], utc=True, errors="coerce")
hourly_df["hour"] = pd.to_datetime(hourly_df["hour"], utc=True, errors="coerce")

print("Train features shape:", train_features.shape)
print("Hourly dataset shape:", hourly_df.shape)

hourly_df.tail()

Train features shape: (20997, 12)
Hourly dataset shape: (23687, 7)


,hour,session_count,total_kwh,hour_of_day,day_of_week,month,is_weekend
23682,2021-09-13 21:00:00+00:00,2,9.000,21,0,9,0
23683,2021-09-13 22:00:00+00:00,1,17.720,22,0,9,0
23684,2021-09-13 23:00:00+00:00,1,2.018,23,0,9,0
23685,2021-09-14 00:00:00+00:00,0,0.000,0,1,9,0
23686,2021-09-14 01:00:00+00:00,1,45.064,1,1,9,0


In [4]:
# Define the feature columns used by the Gradient Boosting model
feature_cols = [
    "hour_of_day",
    "day_of_week",
    "month",
    "is_weekend",
    "lag_1",
    "lag_2",
    "lag_24",
    "lag_168",
    "rolling_mean_24",
]

# Define the target variable
target_col = "session_count"

# Split training features into predictors and target
X_train = train_features[feature_cols]
y_train = train_features[target_col]

# Train the Gradient Boosting model
gradient_boosting_model = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    random_state=42,
)
gradient_boosting_model.fit(X_train, y_train)

print("Gradient Boosting model trained for future forecasting.")

Gradient Boosting model trained for future forecasting.


In [5]:
def build_future_feature_row(timestamp: pd.Timestamp, history_series: pd.Series) -> pd.DataFrame:
    """
    Build one future feature row for the selected timestamp.

    The calendar features come directly from the timestamp.
    The lag and rolling features come from the assumed/predicted history.
    """
    return pd.DataFrame({
        "hour_of_day": [timestamp.hour],
        "day_of_week": [timestamp.dayofweek],
        "month": [timestamp.month],
        "is_weekend": [1 if timestamp.dayofweek in [5, 6] else 0],
        "lag_1": [history_series.iloc[-1]],
        "lag_2": [history_series.iloc[-2]],
        "lag_24": [history_series.iloc[-24]],
        "lag_168": [history_series.iloc[-168]],
        "rolling_mean_24": [history_series.iloc[-24:].mean()],
    })

In [6]:
def recursive_future_forecast(
    model,
    seed_history: pd.Series,
    future_start: pd.Timestamp,
    horizon_hours: int,
) -> pd.DataFrame:
    """
    Forecast future hourly demand recursively.

    seed_history:
        The assumed known hourly history before the future start time

    future_start:
        The first future timestamp to predict

    horizon_hours:
        Number of future hours to predict
    """
    # Copy the seed history so we can append predictions step by step
    history = seed_history.copy().reset_index(drop=True)

    future_rows = []

    for step in range(horizon_hours):
        current_time = future_start + pd.Timedelta(hours=step)

        # Build one feature row using the current timestamp and current history
        X_future = build_future_feature_row(current_time, history)

        # Predict the next hour
        y_pred = model.predict(X_future)[0]

        # Save the future prediction row
        future_rows.append({
            "hour": current_time,
            "predicted_session_count": y_pred,
            "day_of_week": current_time.dayofweek,
            "day_name": current_time.day_name(),
            "hour_of_day": current_time.hour,
            "is_weekend": 1 if current_time.dayofweek in [5, 6] else 0,
        })

        # Append the new prediction so later hours can use it as lag history
        history = pd.concat([history, pd.Series([y_pred])], ignore_index=True)

    return pd.DataFrame(future_rows)

In [7]:
# Use the last 168 hours of the historical dataset as assumed known history
seed_length = 168
seed_history = hourly_df["session_count"].iloc[-seed_length:].reset_index(drop=True)

print("Seed history length:", len(seed_history))
print("Last known historical timestamp:", hourly_df["hour"].iloc[-1])

Seed history length: 168
Last known historical timestamp: 2021-09-14 01:00:00+00:00


In [8]:
# Build a dropdown of possible forecast start dates
# These can be historical continuation or arbitrary future dates for demo purposes
start_options = [
    "2021-09-14 02:00:00+00:00",
    "2021-09-15 00:00:00+00:00",
    "2021-09-20 08:00:00+00:00",
    "2021-10-01 00:00:00+00:00",
    "2026-03-28 00:00:00+00:00",
    "2026-03-28 12:00:00+00:00",
    "2026-03-29 00:00:00+00:00",
]

In [15]:
def demo_future_forecast(start_time="2026-03-28 00:00:00+00:00", horizon_hours=24):
    """
    Interactive toy future forecast demo.

    start_time:
        Choose the first timestamp to forecast

    horizon_hours:
        Number of hours to forecast into the future
    """
    future_start = pd.to_datetime(start_time, utc=True)

    print("Selected forecast start:", future_start)
    print("Day name:", future_start.day_name())
    print("Hour of day:", future_start.hour)
    print("Forecast horizon:", horizon_hours, "hours")
    print()

    # Run recursive future forecasting
    forecast_df = recursive_future_forecast(
        model=gradient_boosting_model,
        seed_history=seed_history,
        future_start=future_start,
        horizon_hours=horizon_hours,
    )

    # Show the forecast table
    display(forecast_df)

    # Plot the recent known history plus the future forecast
    history_hours = hourly_df["hour"].iloc[-seed_length:].reset_index(drop=True)
    history_values = hourly_df["session_count"].iloc[-seed_length:].reset_index(drop=True)

    # Build a relative-hour axis so the plot stays readable even for far-future dates
    history_x = np.arange(-seed_length, 0)
    forecast_x = np.arange(0, horizon_hours)

    plt.figure(figsize=(12, 6))

    # Plot assumed known history using relative past hours
    plt.plot(history_x, history_values, label="Assumed Known History", linewidth=2)

    # Plot future forecast using hours ahead
    plt.plot(
        forecast_x,
        forecast_df["predicted_session_count"],
        label="Future Forecast",
        linewidth=2,
    )

    # Mark the forecast start
    plt.axvline(0, linestyle="--")

    plt.title("Toy Future Forecast Using Gradient Boosting")
    plt.xlabel("Hours Relative to Forecast Start")
    plt.ylabel("Predicted Session Count")
    plt.legend()
    plt.tight_layout()
    plt.show()  

In [16]:
# Interactive controls for the toy future forecast
interact(
    demo_future_forecast,
    start_time=Dropdown(
        options=start_options,
        value="2026-03-28 00:00:00+00:00",
        description="Start"
    ),
    horizon_hours=IntSlider(
        value=24,
        min=6,
        max=48,
        step=6,
        description="Hours"
    ),
);

interactive(children=(Dropdown(description='Start', index=4, options=('2021-09-14 02:00:00+00:00', '2021-09-15…